In [1]:
from langchain_core.prompts import PromptTemplate
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_community.llms import CTransformers
from langchain.chains import RetrievalQA
import chainlit as cl

In [2]:
def load_llm():
    """
    Loading the llama2 model we have installed using CTransformers
    """
    llm = CTransformers(
        model= "llama-2-7b-chat.ggmlv3.q8_0.bin",
        model_type= "llama",
        max_new_tokens = 512,
        temperature = 0.5
    )
    return llm

In [10]:
custom_prompt_template = """Use the following information to answer the users question, if you dont know the answer
just say " I don't know the answer". DO NOT make up answers that are not based on facts. Explain with detailed answers
that are easy to understand

Context: {context}
Question: {question}

Only return the useful aspects of the answer below and nothing else.
Helpful answer:
"""

In [1]:
model_name="TheBloke/Llama-2-7B-Chat-GGML"
model_basename="llama-2-7b-chat.ggmlv3.q8_0.bin"

In [2]:
from huggingface_hub import hf_hub_download

c:\Users\mehul\anaconda3\envs\newenv\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [4]:
model_path=hf_hub_download(repo_id=model_name,filename=model_basename)

c:\Users\mehul\anaconda3\envs\newenv\lib\site-packages\huggingface_hub\file_download.py:140: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\mehul\.cache\huggingface\hub\models--TheBloke--Llama-2-7B-Chat-GGML. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


In [5]:
model_path

'C:\\Users\\mehul\\.cache\\huggingface\\hub\\models--TheBloke--Llama-2-7B-Chat-GGML\\snapshots\\76cd63c351ae389e1d4b91cab2cf470aab11864b\\llama-2-7b-chat.ggmlv3.q8_0.bin'

In [8]:
%pip install ctransformers

  Using cached ctransformers-0.2.27-py3-none-any.whl.metadata (17 kB)
  Using cached py_cpuinfo-9.0.0-py3-none-any.whl.metadata (794 bytes)
Using cached ctransformers-0.2.27-py3-none-any.whl (9.9 MB)
Using cached py_cpuinfo-9.0.0-py3-none-any.whl (22 kB)


In [7]:
def load_llm():
    """
    Loading the llama2 model we have installed using CTransformers
    """
    llm = CTransformers(
        model= "llama-2-7b-chat.ggmlv3.q8_0.bin",
        model_type= "llama",
        max_new_tokens = 512,
        temperature = 0.5
    )
    return llm

In [3]:
def set_custom_prompt():
    """
    Prompt template for QA retrieval for each vector store, we also pass in context and question.
    """
    prompt = PromptTemplate(template= custom_prompt_template, input_variables=['context','question'])
    return prompt

In [5]:
def retrieval_qa_chain(llm,prompt,db):
    """
    Setting up a retrieval-based question-answering chain,
    and returning response
    """
    qa_chain = RetrievalQA.from_chain_type(llm = llm,
                            chain_type = 'stuff',
                            retriever = db.as_retriever(search_kwargs = {'k': 2}),
                            return_source_documents = True,
                            chain_type_kwargs = {'prompt':prompt}
                        )
    
    return qa_chain

In [7]:
DB_FAISS_PATH = r"C:\Users\mehul\edumit\llama2-PDF-Chatbot\vectorstores\db_faiss"

In [8]:
def qa_bot():
    """
    Loading the db and using it in retrieval_qa_chain
    """
    embeddings = HuggingFaceEmbeddings(model_name = 'sentence-transformers/all-MiniLM-L6-v2', model_kwargs = {'device':'cuda'})
    
    db = FAISS.load_local(DB_FAISS_PATH, embeddings)
    llm = load_llm()
    qa_prompt = set_custom_prompt()
    qa = retrieval_qa_chain(llm,qa_prompt,db)
    return qa

In [9]:

def final_result(query):
    qa_result = qa_bot()
    response = qa_result({'query':query})
    return response